# Hypothesis: the initial projection preserves forest performance

We compare the same random forest trained on the original features `X` and on the initial projected representation `X₀`. The practical tolerance is five percentage points of accuracy loss. The test split is used only for the final report.

In [1]:
import warnings
from sklearn.random_projection import DataDimensionalityWarning
warnings.filterwarnings(
    "ignore",
    category=DataDimensionalityWarning,
    message="The number of components is higher than the number of features.*",
)

import sys
from pathlib import Path

notebooks_dir = Path.cwd() / 'notebooks'
if not (notebooks_dir / '_hypothesis_utils.py').exists():
    notebooks_dir = Path.cwd()
sys.path.insert(0, str(notebooks_dir))

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score
from _hypothesis_utils import (
    CLASSIFICATION_SEEDS, classification_data, forest_classifier,
    initial_projection, plot_critical_difference, print_verdict,
)

In [2]:
rows = []
for seed in CLASSIFICATION_SEEDS[:2]:
    X_train, _, X_test, y_train, _, y_test = classification_data(seed)
    p = X_train.shape[1]
    dimension = 20 * p
    print(f'seed={seed}: p={p}, d=20*p={dimension}')
    direct = forest_classifier(seed, n_estimators=100).fit(X_train, y_train)
    direct_score = accuracy_score(y_test, direct.predict(X_test))
    X_train_0, X_test_0, _ = initial_projection(X_train, X_test, dimension, seed)
    projected = forest_classifier(seed, n_estimators=100).fit(X_train_0, y_train)
    projected_score = accuracy_score(y_test, projected.predict(X_test_0))
    rows.append({
        'seed': seed, 'p': p, 'dimension': dimension,
        'accuracy_X': direct_score, 'accuracy_X0': projected_score,
        'loss': direct_score - projected_score,
    })
results = pd.DataFrame(rows)
display(results.round(3))
summary = results.groupby('dimension')['loss'].agg(['mean', 'max']).round(3)
display(summary)
print('Critical-difference diagram omitted: this fixed rule tests one dimension per dataset, so there are no competing dimension configurations.')
tolerance = 0.05
supported = bool((summary['max'] <= tolerance).all())
print_verdict(
    'Initial projection preserves forest performance',
    supported,
    f'maximum observed accuracy loss is {summary["max"].max():.3f}; '
    f'predefined tolerance is {tolerance:.2f}',
)

seed=0: p=120, d=20*p=2400


seed=1: p=120, d=20*p=2400


,seed,p,dimension,accuracy_X,accuracy_X0,loss
0,0,120,2400,0.786,0.816,-0.03
1,1,120,2400,0.834,0.834,0.00


,mean,max
dimension,,
2400,-0.015,0.0


Critical-difference diagram omitted: this fixed rule tests one dimension per dataset, so there are no competing dimension configurations.
Initial projection preserves forest performance: SUPPORTED — maximum observed accuracy loss is 0.000; predefined tolerance is 0.05


## Conclusion

**Hypothesis:** the initial projection preserves random-forest performance within a 0.05 absolute-accuracy tolerance.

**Experiment:** fit matched forests on the original and projected representations across two seeds and target dimensions 512, 2048, and 8192 using the difficult 120-feature classification task.

**Measure:** held-out test accuracy loss, defined as accuracy on `X` minus accuracy on `X₀`; the predefined tolerance was 0.05.

**Result:** not supported in this run: the maximum observed loss was 0.055, slightly above the tolerance.